# 태양풍 속도 예측 — P16: P3 피처 + 다항 릿지회귀

P16 은 P3 의 **피처는 그대로 두고 학습기만 바꾼 대조군**이다. 3×5 코로나홀 격자, 탄도
정렬, 결측 처리, 공식 검증 지표는 P3 와 동일하고, GRU 대신 닫힌 형태의 다항 릿지회귀를
푼다. `USE_*` 토글은 P15 와 이름·의미가 같으므로 두 노트북의 같은 조합은 같은 피처
집합을 재고 있다.

이게 쓸모 있는 이유는 두 가지다.

1. **피처 선별이 몇 초로 끝난다.** 릿지는 정규방정식 한 번이라 GRU 60 epoch 대신
   조합당 수십 초다. P15 에서 어떤 토글을 살릴지 먼저 여기서 훑고, 살아남은 조합만
   GRU 로 확인하면 된다.
2. **신호와 용량을 분리한다.** P16 이 persistence 를 못 이기면 그 피처에는 선형·2차
   범위의 신호가 없다는 뜻이고, P16 은 이기는데 P15 가 못 이기면 학습 쪽 문제다.

P16 은 P15 를 대체하지 않는다. 20 프레임 시계열을 요약통계 몇 개로 접어 버리므로 GRU 가
쓰는 시간 구조 정보를 상당히 버린다. 제출 후보가 아니라 진단 도구로 읽는 게 맞다.

플레어·형태·강도깊이 토글은 `work/cache/p14a_*.npz` 캐시를 필요로 한다. 캐시가 없으면
「P14 캐시 준비」 셀이 같은 폴더의 `p14_extract.py` 를 직접 실행해 만든다.

## 0. 설정

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"),
    Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    searched = "\n".join(f"  - {c}" for c in DATA_ROOT_CANDIDATES if c is not None)
    raise FileNotFoundError("데이터 경로를 찾지 못했습니다:\n" + searched)

WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs_p16"
SUBMISSION_DIR = Path("submission")
for directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 128
CHANNELS = ("193", "211")
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
SCHEDULER_PATIENCE = 3
EARLY_STOP_PATIENCE = 10
NUM_WORKERS = 4
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0
LOSS_MODE = "metric"

# --- 브랜치 스위치 (ablation 용) ---------------------------------------
USE_CNN = False    # 3D CNN 영상 브랜치. P3 기본값은 끔 (과적합 주범)
USE_CH = True      # 코로나홀 격자 피처 브랜치
USE_BALLISTIC = True   # horizon별 탄도 정렬 피처

# --- 코로나홀 추출 (Collin 2025) ---------------------------------------
CH_GRID = (3, 5)          # (위도 구간, 경도 구간). 논문은 4x3 이 timeline RMSE 최적
CH_THRESHOLD_RATIO = 0.45  # 원반 중앙값 대비 이 비율보다 어두우면 코로나홀
DISK_MARGIN = 0.95         # 림 밝아짐(limb brightening) 회피용 반지름 축소
TRANSIT_SPEEDS = (350.0, 500.0, 700.0)   # 탄도 역산에 쓸 가정 속도 (km/s)
AU_KM = 1.496e8

DROPOUT = 0.4
HORIZON_EMBED = 8
AUGMENT = True
AUG_BRIGHTNESS = 0.10
AUG_SHIFT_PIXELS = 6
AUG_NOISE_STD = 0.02
AUG_ERASE_PROB = 0.3
AUG_CH_NOISE = 0.05        # CH 피처에 주는 곱셈 노이즈

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA Unavailable")

print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())
print(f"branches: CNN={USE_CNN} CH={USE_CH} BALLISTIC={USE_BALLISTIC}")

# ============================= P16 설정 ====================================
# P15 와 같은 토글. 같은 조합끼리 직접 비교할 수 있게 이름을 맞춘다.
USE_FLARE = False
USE_CH_SHAPE = False
USE_ROTATION = False
USE_INTENSITY_DEPTH = False
P16_CACHE_VERSION = "p14a"

# 핵심 피처에만 다항 전개를 건다. 1 이면 순수 선형회귀가 된다.
P16_DEGREE = 2
# 릿지 계수는 validation 공식 RMSE 로 고르고, 12 horizon 이 하나의 alpha 를 공유한다.
P16_ALPHAS = (0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0, 30000.0)
# 20 프레임 시계열을 접는 방법. 줄이면 설계행렬이 그만큼 좁아진다.
P16_SUMMARIES = ("last", "mean", "mean4", "slope")
P16_CHUNK = 4096          # 시퀀스 요약을 나눠 처리하는 행 수 (메모리 상한용)
P16_REQUESTED_SWITCHES = {
    "flare": USE_FLARE,
    "shape": USE_CH_SHAPE,
    "rotation": USE_ROTATION,
    "intensity_depth": USE_INTENSITY_DEPTH,
}

## 1. 데이터 로드 · 결측 처리

In [ ]:
IMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]
WIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]
TARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]
HORIZONS = np.arange(1, 13) * 6

train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
test_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")

assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()
assert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()
assert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)
assert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert not any(column.startswith("target_") for column in test_inputs.columns)


def forward_fill_rows(values):
    valid = np.isfinite(values)
    positions = np.where(valid, np.arange(values.shape[1])[None, :], 0)
    np.maximum.accumulate(positions, axis=1, out=positions)
    rows = np.arange(values.shape[0])[:, None]
    return np.where(valid.any(axis=1, keepdims=True), values[rows, positions], values)


def fill_wind(frame, fallback):
    values = frame[WIND_COLUMNS].to_numpy(np.float32)
    valid = np.isfinite(values).astype(np.float32)
    filled = forward_fill_rows(values)
    filled = forward_fill_rows(filled[:, ::-1])[:, ::-1]
    filled = np.where(np.isfinite(filled), filled, fallback)
    return np.ascontiguousarray(filled), np.ascontiguousarray(valid)


WIND_FALLBACK = float(np.nanmedian(train_inputs[WIND_COLUMNS].to_numpy(np.float32)))
train_wind, train_wind_valid = fill_wind(train_inputs, WIND_FALLBACK)
val_wind, val_wind_valid = fill_wind(val_inputs, WIND_FALLBACK)
test_wind, test_wind_valid = fill_wind(test_inputs, WIND_FALLBACK)

train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
assert np.isfinite(train_targets).all() and np.isfinite(val_targets).all()

print("samples:", len(train_inputs), len(val_inputs), len(test_inputs))
print(f"train wind mean={train_wind.mean():.1f} target mean={train_targets.mean():.1f}")

## 2. 이미지 memory-map cache

In [ ]:
def prepare_image_memmap(split, inputs):
    image_root = DATA_ROOT / split
    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_images.npy"
    metadata_path = cache_root / f"{split}_metadata.json"
    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    expected = {"image_size": IMAGE_SIZE, "channels": list(CHANNELS), "filenames": filenames}
    shape = (len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE)

    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = (metadata == expected and cached.shape == shape
                     and cached.dtype == np.uint8)
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False

    if not valid:
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        metadata_temp = metadata_path.with_name(metadata_path.name + f".partial.{os.getpid()}")
        resized = np.lib.format.open_memmap(array_temp, mode="w+", dtype=np.uint8, shape=shape)
        resampling = Image.Resampling.BILINEAR
        for index, filename in enumerate(filenames):
            for channel_index, channel in enumerate(CHANNELS):
                with Image.open(image_root / channel / filename) as image:
                    resized[index, channel_index] = np.asarray(
                        image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), resampling),
                        dtype=np.uint8)
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)
        resized.flush()
        del resized
        metadata_temp.write_text(json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8")
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created cache: {array_path.resolve()}")
    else:
        print(f"reusing cache: {array_path.resolve()}")

    image_array = np.load(array_path, mmap_mode="r")
    return image_array, {name: i for i, name in enumerate(filenames)}


train_image_array, train_image_index = prepare_image_memmap("train", train_inputs)
val_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)
test_image_array, test_image_index = prepare_image_memmap("test", test_inputs)
print("고유 이미지:", len(train_image_index), len(val_image_index), len(test_image_index))

## 3. 태양 원반 검출 · 코로나홀 격자 면적 추출 — **P3 핵심**

Collin 2025 의 접근을 규정 내에서 재현합니다.

1. **원반 검출** — train 평균 영상에서 밝은 영역의 중심/반지름을 추정. 림 밝아짐을 피해 반지름을 5% 축소
2. **코로나홀 마스크** — 원반 내부 중앙값 대비 `CH_THRESHOLD_RATIO` 보다 어두우면서
   **193과 211 두 채널 모두에서 어두운** 픽셀만 인정 (논문의 2채널 결합 아이디어)
3. **격자 binning** — 원반을 (위도 × 경도) 격자로 나눠 셀별 CH 면적 비율 산출

결과는 이미지당 `CH_GRID` 개 실수뿐이라 저장 용량이 무시할 수준이고, **저차원이라 과적합에 강합니다.**

In [ ]:
def detect_disk(image_array, sample_count=400):
    indexes = np.unique(np.linspace(0, len(image_array) - 1, sample_count).astype(int))
    mean_image = np.asarray(image_array[indexes], dtype=np.float64).mean(axis=(0, 1))
    # 배경은 어둡고 원반은 밝습니다. 단순 임계로 원반 픽셀을 잡습니다.
    mask = mean_image > mean_image.max() * 0.15
    ys, xs = np.nonzero(mask)
    center_y, center_x = float(ys.mean()), float(xs.mean())
    radius = float(np.sqrt(mask.sum() / np.pi))
    return center_y, center_x, radius, mean_image


DISK_Y, DISK_X, DISK_R, MEAN_IMAGE = detect_disk(train_image_array)
EFFECTIVE_R = DISK_R * DISK_MARGIN
print(f"원반 검출: center=({DISK_Y:.1f}, {DISK_X:.1f}) radius={DISK_R:.1f}px "
      f"(유효 {EFFECTIVE_R:.1f}px, 프레임의 {2*DISK_R/IMAGE_SIZE:.0%})")

grid_y, grid_x = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE].astype(np.float64)
radius_map = np.sqrt((grid_y - DISK_Y) ** 2 + (grid_x - DISK_X) ** 2)
DISK_MASK = radius_map <= EFFECTIVE_R

# 원반 외접 사각형을 (위도, 경도) 격자로 분할
n_lat, n_lon = CH_GRID
lat_edge = np.clip(((grid_y - (DISK_Y - EFFECTIVE_R)) / (2 * EFFECTIVE_R) * n_lat), 0, n_lat - 1e-6)
lon_edge = np.clip(((grid_x - (DISK_X - EFFECTIVE_R)) / (2 * EFFECTIVE_R) * n_lon), 0, n_lon - 1e-6)
CELL_ID = (lat_edge.astype(np.int64) * n_lon + lon_edge.astype(np.int64))
CELL_ID_FLAT = CELL_ID[DISK_MASK]
N_CELLS = n_lat * n_lon
CELL_COUNTS = np.bincount(CELL_ID_FLAT, minlength=N_CELLS).astype(np.float32)
CELL_COUNTS = np.maximum(CELL_COUNTS, 1.0)
# 경도 중앙 열(자오선)과 적도 위도대의 셀 인덱스
CENTRAL_LON = n_lon // 2
EQUATOR_LAT = n_lat // 2
CENTRAL_CELL = EQUATOR_LAT * n_lon + CENTRAL_LON
print(f"격자 {n_lat}x{n_lon} = {N_CELLS} 셀, 원반 픽셀 {int(DISK_MASK.sum()):,}개, "
      f"중앙자오선 셀 index={CENTRAL_CELL}")


def compute_ch_grid(image_array, chunk=256):
    # 반환: (n_images, N_CELLS) 셀별 코로나홀 면적 비율
    result = np.zeros((len(image_array), N_CELLS), dtype=np.float32)
    onehot = np.zeros((len(CELL_ID_FLAT), N_CELLS), dtype=np.float32)
    onehot[np.arange(len(CELL_ID_FLAT)), CELL_ID_FLAT] = 1.0
    for start in range(0, len(image_array), chunk):
        block = np.asarray(image_array[start:start + chunk], dtype=np.float32)
        on_disk = block[:, :, DISK_MASK]                       # (n, 2, npix)
        median = np.median(on_disk, axis=2, keepdims=True)     # (n, 2, 1)
        dark = on_disk < (CH_THRESHOLD_RATIO * median)
        # 193 과 211 두 채널 모두에서 어두운 픽셀만 코로나홀로 인정
        coronal_hole = np.logical_and(dark[:, 0], dark[:, 1]).astype(np.float32)
        result[start:start + chunk] = (coronal_hole @ onehot) / CELL_COUNTS
    return result


def cached_ch_grid(split, image_array):
    path = CACHE_ROOT / f"ch_{split}_{n_lat}x{n_lon}_{CH_THRESHOLD_RATIO}_{IMAGE_SIZE}.npy"
    if path.exists():
        grid = np.load(path)
        if grid.shape == (len(image_array), N_CELLS):
            print(f"reusing CH cache: {path.name}")
            return grid
    grid = compute_ch_grid(image_array)
    np.save(path, grid)
    print(f"created CH cache: {path.name}  shape={grid.shape}")
    return grid


train_ch = cached_ch_grid("train", train_image_array)
val_ch = cached_ch_grid("validation", val_image_array)
test_ch = cached_ch_grid("test", test_image_array)

print(f"\nCH 면적 비율 — train 전체 평균 {train_ch.mean():.4f}, "
      f"중앙자오선 셀 평균 {train_ch[:, CENTRAL_CELL].mean():.4f}")

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(MEAN_IMAGE, cmap="gray")
circle = plt.Circle((DISK_X, DISK_Y), EFFECTIVE_R, fill=False, color="red", linewidth=1.5)
axes[0].add_patch(circle)
axes[0].set_title("train 평균 영상 + 검출된 원반")
sample_image = np.asarray(train_image_array[0], dtype=np.float32)
sample_dark = sample_image < (CH_THRESHOLD_RATIO * np.median(sample_image[:, DISK_MASK], axis=1)[:, None, None])
axes[1].imshow(np.logical_and(sample_dark[0], sample_dark[1]) & DISK_MASK, cmap="gray")
axes[1].set_title("코로나홀 마스크 예시 (193 AND 211)")
axes[2].imshow(train_ch[:400].T, aspect="auto", cmap="viridis")
axes[2].set_title("셀별 CH 면적 (앞 400시점)")
axes[2].set_xlabel("time index"); axes[2].set_ylabel("cell")
for axis in axes[:2]:
    axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout(); plt.show()

## 4. 정규화 통계 · 탄도 정렬 인덱스

**탄도 정렬**: horizon $h$ 의 타깃이 발원한 시각은 $T_0 + h - \tau$, $\tau = \mathrm{1AU}/v$.
가정 속도 $v \in \{350, 500, 700\}$ km/s 각각에 대해 윈도우 내 (실수) 인덱스를 계산해 둡니다.
윈도우를 벗어나면 양 끝으로 clamp 합니다 (고속풍 + 장기 horizon 은 아직 관측되지 않은 영역에서 발원).

In [ ]:
def compute_image_stats(array, chunk=256):
    total = np.zeros(len(CHANNELS), np.float64)
    total_square = np.zeros(len(CHANNELS), np.float64)
    count = 0
    for start in range(0, len(array), chunk):
        block = np.asarray(array[start:start + chunk], dtype=np.float64) / 255.0
        total += block.sum(axis=(0, 2, 3))
        total_square += (block ** 2).sum(axis=(0, 2, 3))
        count += block.shape[0] * block.shape[2] * block.shape[3]
    mean = total / count
    return mean.astype(np.float32), np.sqrt(
        np.maximum(total_square / count - mean ** 2, 1e-12)).astype(np.float32)


IMAGE_MEAN, IMAGE_STD = compute_image_stats(train_image_array)
WIND_MEAN = float(train_wind.mean())
WIND_STD = float(train_wind.std() + 1e-6)
DIFF_STD = float(np.diff(train_wind, axis=1, prepend=train_wind[:, :1]).std() + 1e-6)
CH_MEAN = train_ch.mean(axis=0).astype(np.float32)
CH_STD = (train_ch.std(axis=0) + 1e-8).astype(np.float32)

train_residual = train_targets - train_wind[:, -1:]
RESIDUAL_MEAN = train_residual.mean(axis=0).astype(np.float32)
RESIDUAL_STD = (train_residual.std(axis=0) + 1e-6).astype(np.float32)
CLIP_LOW = float(train_targets.min() * 0.95)
CLIP_HIGH = float(train_targets.max() * 1.05)


def ballistic_indices():
    # (12, n_speeds) 윈도우 내 실수 인덱스. 인덱스 19 가 마지막 관측 시점 T0.
    table = np.zeros((12, len(TRANSIT_SPEEDS)), dtype=np.float32)
    for horizon_index in range(12):
        lead_hours = (horizon_index + 1) * 6.0
        for speed_index, speed in enumerate(TRANSIT_SPEEDS):
            transit_hours = AU_KM / speed / 3600.0
            table[horizon_index, speed_index] = np.clip(
                19.0 + (lead_hours - transit_hours) / 6.0, 0.0, 19.0)
    return table


BALLISTIC_INDEX = ballistic_indices()
N_SPEEDS = len(TRANSIT_SPEEDS)


def image_index_matrix(inputs, image_index):
    return np.asarray([
        [image_index[name] for name in row]
        for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
    ], dtype=np.int32)


def compute_ballistic(ch_grid, indexes):
    # 중앙자오선 셀 시계열을 탄도 역산 인덱스에서 선형보간 -> (n, 12, n_speeds)
    central = ch_grid[indexes][:, :, CENTRAL_CELL]
    lower = np.floor(BALLISTIC_INDEX).astype(np.int64)
    upper = np.minimum(lower + 1, 19)
    weight = (BALLISTIC_INDEX - lower).astype(np.float32)
    return (central[:, lower] * (1.0 - weight) + central[:, upper] * weight).astype(np.float32)


# 탄도 피처 정규화 통계도 반드시 train split 에서만 산출합니다.
_train_ballistic = compute_ballistic(train_ch, image_index_matrix(train_inputs, train_image_index))
BALLISTIC_MEAN = float(_train_ballistic.mean())
BALLISTIC_STD = float(_train_ballistic.std() + 1e-8)

print("탄도 역산 — 가정 속도별 전달 시간:")
for speed in TRANSIT_SPEEDS:
    print(f"  v={speed:5.0f} km/s -> tau = {AU_KM / speed / 3600.0:5.1f} h "
          f"({AU_KM / speed / 86400.0:.2f} 일)")
print("\nhorizon별 근원 시점 인덱스 (19 = 마지막 관측):")
print(pd.DataFrame(BALLISTIC_INDEX, index=[f"{h}h" for h in HORIZONS],
                   columns=[f"v={int(v)}" for v in TRANSIT_SPEEDS]).round(2))
print(f"\nresidual std by horizon: {np.round(RESIDUAL_STD, 1)}")
print(f"clip range: [{CLIP_LOW:.1f}, {CLIP_HIGH:.1f}] km/s")

## 5. P14 물리 피처 캐시 준비 (없으면 여기서 생성)

In [ ]:
# ===================== P16: P14 물리 피처 캐시 준비 =========================
# work/cache/p14a_{train,validation,test}.npz 가 없으면 이 셀이 직접 만든다.
# 노트북과 같은 폴더에 p14_extract.py 와 p11_extract.py 가 함께 있어야 한다.
# 캐시가 이미 있으면 아무 일도 하지 않고 즉시 넘어간다.
import subprocess
import sys

P16_AUTO_EXTRACT = True     # False 로 두면 캐시가 없어도 추출하지 않는다
P16_EXTRACT_WORKERS = 4     # 운영진 공지: 워커 4개를 넘기지 말 것
P16_EXTRACT_SPLITS = ("train", "validation", "test")


def p16_cache_path(split):
    return CACHE_ROOT / f"{P16_CACHE_VERSION}_{split}.npz"


def p16_missing_splits():
    return [split for split in P16_EXTRACT_SPLITS if not p16_cache_path(split).exists()]


def p16_run_extract(splits):
    """p14_extract.py 를 자식 프로세스로 돌리고 진행 로그를 그대로 흘린다."""
    scripts = [Path("p14_extract.py"), Path("p11_extract.py")]
    absent = [str(script.resolve()) for script in scripts if not script.exists()]
    if absent:
        print("추출 스크립트가 없다:")
        for item in absent:
            print("   ", item)
        print("  -> 이 노트북과 같은 폴더에 p11_extract.py 와 p14_extract.py 를 올린 뒤 다시 실행할 것")
        return False

    command = [sys.executable, "-u", "p14_extract.py",
               "--workers", str(P16_EXTRACT_WORKERS),
               "--splits", ",".join(splits),
               "--cache", str(CACHE_ROOT)]
    # 자식이 데이터 경로를 다시 추측하지 않도록 노트북이 찾은 경로를 그대로 넘긴다.
    environment = dict(os.environ, SW_DATA_ROOT=str(DATA_ROOT.resolve()))
    print("실행:", " ".join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               env=environment, text=True, encoding="utf-8",
                               errors="replace", bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status != 0:
        print(f"p14_extract.py 가 실패했다 (exit {status}).", flush=True)
    return status == 0


_p16_missing = p16_missing_splits()
if not _p16_missing:
    print("P14 캐시 확인 완료:")
    for split in P16_EXTRACT_SPLITS:
        _path = p16_cache_path(split)
        print(f"  {split:<10} {_path.resolve()} ({_path.stat().st_size / 1024 ** 2:.0f} MB)")
elif not P16_AUTO_EXTRACT:
    print("P14 캐시 없음:", ", ".join(_p16_missing), "| P16_AUTO_EXTRACT=False 이므로 건너뛴다")
else:
    print("P14 캐시 없음 ->", ", ".join(_p16_missing), "추출을 시작한다.")
    print("전체 split 은 수십 분에서 몇 시간까지 걸린다. 터미널을 쓸 수 있으면 대신")
    print("  nohup python p14_extract.py --workers 4 > extract.log 2>&1 &")
    print("로 돌리고 이 셀을 다시 실행하는 편이 낫다.", flush=True)
    p16_run_extract(_p16_missing)
    _p16_missing = p16_missing_splits()
    print("아직 없는 split:", ", ".join(_p16_missing) if _p16_missing else "없음")

## 6. 물리 피처 로드 — P3 파일 순서로 재매핑

In [ ]:
# ===================== P16: 물리 피처 로드 (P15 와 동일) ====================
# p14a 의 파일 순서는 시간 사슬 순서이고 P3 memmap 은 파일명 정렬 순서다.
# 반드시 filename 으로 재매핑한다. 배열 위치가 우연히 같다고 가정하지 않는다.

P16_CACHE_KEYS = ("area", "depth_max", "depth_sum", "flare_area", "flare_power", "shape")


def p16_load(split, image_index):
    path = CACHE_ROOT / f"{P16_CACHE_VERSION}_{split}.npz"
    if not path.exists():
        print(f"  ⚠️ 캐시 파일 없음: {path.resolve()}")
        return None
    with np.load(path) as blob:
        if any(key not in blob for key in P16_CACHE_KEYS) or "files" not in blob:
            print(f"  ⚠️ {path.name}: P14 키 없음 -> p14_extract.py 를 실행할 것")
            return None
        source_index = {str(name): i for i, name in enumerate(blob["files"])}
        names = list(image_index)
        if set(names) != set(source_index):
            print(f"  ⚠️ {path.name}: filename 집합 불일치 -> P14 피처를 쓰지 않는다")
            return None
        order = np.asarray([source_index[name] for name in names], np.int64)
        return {key: np.asarray(blob[key][order], np.float32) for key in P16_CACHE_KEYS}


P16 = {
    "train": p16_load("train", train_image_index),
    "validation": p16_load("validation", val_image_index),
    "test": p16_load("test", test_image_index),
}
P16_AVAILABLE = all(value is not None for value in P16.values())
if not P16_AVAILABLE:
    print("P14 cache unavailable: flare/shape/intensity-depth disabled; rotation is still available.")
    print("  해결: 노트북 폴더에서 아래를 실행한 다음 위의 「P14 캐시 준비」 셀부터 다시 실행할 것")
    print("    python p14_extract.py --workers 4")
    USE_FLARE = USE_CH_SHAPE = USE_INTENSITY_DEPTH = False
else:
    print("P14 cache available: P3-compatible feature remapping complete.")


def p16_aggregate(fine, channels=1):
    """P14 12x30 fine grid -> P3 3x5 grid. 값은 원반 면적비이므로 합으로 접는다."""
    block = fine.reshape(len(fine), channels, 12, 30)
    return block.reshape(len(fine), channels, 3, 4, 5, 6).sum(axis=(3, 5)).reshape(len(fine), -1)


def p16_static(split, base_ch):
    """이미지 1장당 피처 (n_images, F). 첫 15 열은 언제나 P3 원본 격자다."""
    pieces = [base_ch.astype(np.float32)]
    if not P16_AVAILABLE:
        return np.concatenate(pieces, axis=1)
    cached = P16[split]
    if USE_FLARE:
        pieces += [p16_aggregate(cached["flare_area"]), p16_aggregate(cached["flare_power"])]
    if USE_CH_SHAPE:
        pieces += [p16_aggregate(cached["depth_max"]), p16_aggregate(cached["depth_sum"]),
                   cached["shape"]]
    if USE_INTENSITY_DEPTH:
        pieces += [p16_aggregate(cached["area"][:, :3], channels=3),
                   p16_aggregate(cached["depth_max"]), p16_aggregate(cached["depth_sum"])]
    return np.concatenate(pieces, axis=1).astype(np.float32)


def p16_rotation(raw_ch):
    """P3 3x5 격자만으로 만드는 자전 상태. P15 의 p15_rotation 과 같은 정의다."""
    area = raw_ch.reshape(len(raw_ch), 20, 3, 5).sum(axis=2)
    position = np.arange(5, dtype=np.float32) - 2.0
    centrality = 1.0 - np.abs(position) / 2.0
    core = np.einsum("btl,l->bt", area, centrality)
    side = np.einsum("btl,l->bt", area, position / 2.0)
    d_core = np.diff(core, axis=1, prepend=core[:, :1])
    d_side = np.diff(side, axis=1, prepend=side[:, :1])
    return np.stack((core, side, d_core, d_side), axis=-1).astype(np.float32)


P16_STATIC = {
    "train": p16_static("train", train_ch),
    "validation": p16_static("validation", val_ch),
    "test": p16_static("test", test_ch),
}
P16_INDEXES = {
    "train": image_index_matrix(train_inputs, train_image_index),
    "validation": image_index_matrix(val_inputs, val_image_index),
    "test": image_index_matrix(test_inputs, test_image_index),
}
P16_RAW_CH = {"train": train_ch, "validation": val_ch, "test": test_ch}
P16_WIND = {"train": train_wind, "validation": val_wind, "test": test_wind}
P16_SPLITS = ("train", "validation", "test")


def p16_sequence(split, rows=slice(None)):
    """P15 가 GRU 에 넣는 것과 같은 (n, 20, F) 입력 시퀀스. rows 로 잘라 쓸 수 있다."""
    indexes = P16_INDEXES[split][rows]
    sequence = P16_STATIC[split][indexes]
    if USE_ROTATION:
        sequence = np.concatenate((sequence, p16_rotation(P16_RAW_CH[split][indexes])), axis=2)
    return sequence


P16_ACTIVE_SWITCHES = {
    "flare": USE_FLARE, "shape": USE_CH_SHAPE,
    "rotation": USE_ROTATION, "intensity_depth": USE_INTENSITY_DEPTH,
}
P16_FEATURE_DIM = P16_STATIC["train"].shape[1] + (4 if USE_ROTATION else 0)
P16_EXPECTED_DIM = (15 + (30 if USE_FLARE else 0) + (36 if USE_CH_SHAPE else 0)
                    + (75 if USE_INTENSITY_DEPTH else 0) + (4 if USE_ROTATION else 0))
assert P16_FEATURE_DIM == P16_EXPECTED_DIM, (P16_FEATURE_DIM, P16_EXPECTED_DIM)
print("P16 requested switches:", P16_REQUESTED_SWITCHES)
print("P16 active switches:   ", P16_ACTIVE_SWITCHES)
if P16_REQUESTED_SWITCHES != P16_ACTIVE_SWITCHES:
    print("WARNING: 캐시가 없어 일부 토글이 꺼졌다. p14_extract.py 를 돌리고 다시 실행할 것.")
print(f"P16 프레임당 피처 F = {P16_FEATURE_DIM}")

## 7. 지표 — P3 와 동일한 공식 RMSE

In [ ]:
def official_rmse(y_true, y_pred):
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def pooled_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def metrics_by_horizon(y_true, y_pred, persistence=None):
    rows = []
    for index in range(12):
        actual, predicted = y_true[:, index], y_pred[:, index]
        error = predicted - actual
        denominator = np.std(actual) * np.std(predicted)
        row = {"horizon_h": int(HORIZONS[index]),
               "rmse": float(np.sqrt(np.mean(error ** 2))),
               "mae": float(np.mean(np.abs(error))),
               "corr": float(np.corrcoef(actual, predicted)[0, 1]) if denominator > 0 else np.nan}
        if persistence is not None:
            row["persistence_rmse"] = float(np.sqrt(np.mean((persistence[:, index] - actual) ** 2)))
            row["gain"] = row["persistence_rmse"] - row["rmse"]
        rows.append(row)
    return pd.DataFrame(rows)


val_persistence = np.repeat(val_wind[:, -1:], 12, axis=1).astype(np.float64)
persistence_score, persistence_per_horizon = official_rmse(val_targets, val_persistence)
print(f"[기준선] persistence 공식 RMSE = {persistence_score:.3f} km/s")
print("horizon별:", np.round(persistence_per_horizon, 1))

## 8. 설계행렬 — core 다항 전개 + 넓은 선형 블록

In [ ]:
# ===================== P16: 설계행렬 ======================================
# 다항 전개를 전부에 걸면 항이 폭발한다 (F=160 이면 요약 후 640열, 2차만 205,000 항).
# 그래서 두 층으로 나눈다.
#   core   : 물리적으로 상호작용이 의미 있는 소수 피처. 여기에만 다항 전개를 건다.
#   linear : 나머지 넓은 CH/물리 피처. 1차항으로만 붙인다.
# core 구성은 토글과 무관하게 고정이라 조합을 바꿔도 비교축이 흔들리지 않는다.

from itertools import combinations_with_replacement

_P16_TIME = np.arange(20, dtype=np.float64) - 9.5
_P16_TIME_DENOMINATOR = float((_P16_TIME ** 2).sum())

# P3 cell 12 에 있던 바람 통계. P16 은 그 셀(Dataset)을 쓰지 않으므로 여기 다시 둔다.
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _P16_TIME / _P16_TIME_DENOMINATOR
    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),
                     wind.max(axis=1), slope, last - mean4,
                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float64)


def p16_summaries(sequence):
    """(n, 20, F) -> (n, len(P16_SUMMARIES) * F). summary-major, feature-minor 순서."""
    sequence = sequence.astype(np.float64)
    pieces = []
    for name in P16_SUMMARIES:
        if name == "last":
            pieces.append(sequence[:, -1, :])
        elif name == "mean":
            pieces.append(sequence.mean(axis=1))
        elif name == "mean4":
            pieces.append(sequence[:, -4:, :].mean(axis=1))
        elif name == "slope":
            centered = sequence - sequence.mean(axis=1, keepdims=True)
            pieces.append(np.tensordot(centered, _P16_TIME, axes=([1], [0]))
                          / _P16_TIME_DENOMINATOR)
        else:
            raise ValueError(f"알 수 없는 요약 {name}")
    return np.concatenate(pieces, axis=1)


def p16_summarize_split(split):
    """토글을 다 켜면 (n, 20, 160) 이 1GB 를 넘는다. 행으로 잘라 상한을 건다."""
    total = len(P16_INDEXES[split])
    parts = [p16_summaries(p16_sequence(split, slice(start, start + P16_CHUNK)))
             for start in range(0, total, P16_CHUNK)]
    return np.concatenate(parts, axis=0)


def p16_fold3(series):
    """(n, 20) -> (n, 3): last, mean, slope. core 는 P16_SUMMARIES 설정과 무관하다."""
    series = series.astype(np.float64)
    centered = series - series.mean(axis=1, keepdims=True)
    return np.stack([series[:, -1], series.mean(axis=1),
                     centered @ _P16_TIME / _P16_TIME_DENOMINATOR], axis=1)


def p16_summary_names(feature_names):
    return [f"{name}[{summary}]" for summary in P16_SUMMARIES for name in feature_names]


def p16_feature_names():
    names = [f"ch{cell:02d}" for cell in range(N_CELLS)]
    if USE_FLARE:
        names += [f"flare_area{c:02d}" for c in range(N_CELLS)]
        names += [f"flare_power{c:02d}" for c in range(N_CELLS)]
    if USE_CH_SHAPE:
        names += [f"theta_max{c:02d}" for c in range(N_CELLS)]
        names += [f"theta_sum{c:02d}" for c in range(N_CELLS)]
        names += ["largest_frac", "component_count", "compactness",
                  "elongation", "boundary_density", "core_fraction"]
    if USE_INTENSITY_DEPTH:
        for level in ("dark30", "dark45", "dark60"):
            names += [f"{level}_{c:02d}" for c in range(N_CELLS)]
        names += [f"d_theta_max{c:02d}" for c in range(N_CELLS)]
        names += [f"d_theta_sum{c:02d}" for c in range(N_CELLS)]
    if USE_ROTATION:
        names += ["rot_core", "rot_side", "rot_d_core", "rot_d_side"]
    return names


# ---- core: 바람 통계 9 + CH 총면적 3 + 중앙자오선 3 + horizon 별 탄도 3 = 18 ----
P16_CORE_NAMES = (
    [f"wind_{name}" for name in STAT_NAMES]
    + ["ch_total_last", "ch_total_mean", "ch_total_slope"]
    + ["ch_central_last", "ch_central_mean", "ch_central_slope"]
    + [f"ballistic_v{int(speed)}" for speed in TRANSIT_SPEEDS]
)


def p16_core_static(split):
    """horizon 과 무관한 core 부분. 탄도 3열은 뒤에서 horizon 별로 붙인다."""
    base = P16_RAW_CH[split][P16_INDEXES[split]]      # (n, 20, N_CELLS) 원본 P3 격자
    return np.concatenate([build_wind_stats(P16_WIND[split]),
                           p16_fold3(base.sum(axis=2)),
                           p16_fold3(base[:, :, CENTRAL_CELL])], axis=1)


def p16_standardize(matrix, statistics=None):
    """train 에서만 통계를 내고, 분산이 0 인 열은 아예 버린다 (전부 0 인 플레어 셀 등)."""
    if statistics is None:
        deviation = matrix.std(axis=0)
        keep = deviation > 1e-12
        statistics = (keep, matrix[:, keep].mean(axis=0), deviation[keep])
    keep, mean, deviation = statistics
    return (matrix[:, keep] - mean) / deviation, statistics


def p16_polynomial(matrix, degree):
    """상수항 없는 다항 전개. 절편은 따로 두므로 1 로만 된 열을 만들지 않는다."""
    if degree <= 1:
        return matrix
    columns = [matrix]
    for order in range(2, degree + 1):
        for combination in combinations_with_replacement(range(matrix.shape[1]), order):
            columns.append(matrix[:, combination].prod(axis=1, keepdims=True))
    return np.concatenate(columns, axis=1)


def p16_polynomial_names(names, degree):
    terms = list(names)
    for order in range(2, degree + 1):
        for combination in combinations_with_replacement(range(len(names)), order):
            terms.append("*".join(names[index] for index in combination))
    return terms


# ---- 넓은 선형 블록 ----------------------------------------------------------
P16_FEATURE_NAMES = p16_feature_names()
_linear_raw = {split: p16_summarize_split(split) for split in P16_SPLITS}
P16_LINEAR = {}
P16_LINEAR["train"], P16_LINEAR_STATS = p16_standardize(_linear_raw["train"])
for split in ("validation", "test"):
    P16_LINEAR[split], _ = p16_standardize(_linear_raw[split], P16_LINEAR_STATS)
P16_LINEAR_NAMES = [name for name, keep
                    in zip(p16_summary_names(P16_FEATURE_NAMES), P16_LINEAR_STATS[0]) if keep]
_dropped = int((~P16_LINEAR_STATS[0]).sum())
del _linear_raw

# ---- core: horizon 별 탄도 3열을 붙여 표준화해 둔다 ---------------------------
_core_static = {split: p16_core_static(split) for split in P16_SPLITS}
_ballistic = {split: compute_ballistic(P16_RAW_CH[split], P16_INDEXES[split])
              for split in P16_SPLITS}                       # (n, 12, n_speeds)

P16_CORE, P16_CORE_STATS = {}, {}
for horizon in range(12):
    raw = {split: np.concatenate(
        [_core_static[split], _ballistic[split][:, horizon, :].astype(np.float64)], axis=1)
        for split in P16_SPLITS}
    P16_CORE[("train", horizon)], P16_CORE_STATS[horizon] = p16_standardize(raw["train"])
    for split in ("validation", "test"):
        P16_CORE[(split, horizon)], _ = p16_standardize(raw[split], P16_CORE_STATS[horizon])
del _core_static, _ballistic

_core_kept = int(P16_CORE_STATS[0][0].sum())
P16_TERM_COUNT = len(p16_polynomial_names([str(i) for i in range(_core_kept)], P16_DEGREE))
P16_WIDTH = P16_TERM_COUNT + P16_LINEAR["train"].shape[1]
print(f"core {_core_kept}열 -> {P16_DEGREE}차 전개 {P16_TERM_COUNT}항")
print(f"linear 블록 {P16_LINEAR['train'].shape[1]}열 "
      f"(요약 {len(P16_SUMMARIES)} x F {P16_FEATURE_DIM} 중 상수열 {_dropped}개 제거)")
print(f"설계행렬 폭 D = {P16_WIDTH}, train 표본 n = {len(train_targets):,} "
      f"(n/D = {len(train_targets) / P16_WIDTH:.1f})")
if len(train_targets) < 5 * P16_WIDTH:
    print("  주의: n/D 가 5 미만이다. P16_DEGREE 를 낮추거나 P16_SUMMARIES 를 줄일 것.")

## 9. 릿지 적합 · alpha 선택

In [ ]:
# ===================== P16: 릿지 적합 · alpha 선택 =========================
# horizon 마다 별도 회귀를 푼다. 탄도 피처가 horizon 별로 다르기 때문이다.
# 목표는 P3 와 같은 잔차 = target - 마지막 관측 풍속.
# 설계행렬은 열마다 train 평균이 0 이라 절편은 잔차 평균 하나로 끝난다.
# alpha 는 12 horizon 이 공유한다. horizon 마다 따로 고르면 validation 에 더 과적합된다.

P16_RESIDUAL = train_targets - train_wind[:, -1:]


def p16_design(split, horizon, statistics=None):
    expanded = p16_polynomial(P16_CORE[(split, horizon)], P16_DEGREE)
    expanded, statistics = p16_standardize(expanded, statistics)
    return np.concatenate([expanded, P16_LINEAR[split]], axis=1), statistics


def p16_ridge(gram, moment, alpha):
    regularized = gram.copy()
    regularized.flat[:: gram.shape[0] + 1] += alpha
    return np.linalg.solve(regularized, moment)


P16_GRAM, P16_MOMENT, P16_INTERCEPT, P16_POLY_STATS, P16_VAL_DESIGN = {}, {}, {}, {}, {}
_started = time.perf_counter()
for horizon in range(12):
    design, P16_POLY_STATS[horizon] = p16_design("train", horizon)
    residual = P16_RESIDUAL[:, horizon].astype(np.float64)
    P16_INTERCEPT[horizon] = float(residual.mean())
    P16_GRAM[horizon] = design.T @ design
    P16_MOMENT[horizon] = design.T @ (residual - P16_INTERCEPT[horizon])
    del design
    # validation 설계행렬은 alpha 탐색 내내 재사용하므로 들고 있는다 (표본이 적다).
    P16_VAL_DESIGN[horizon], _ = p16_design("validation", horizon, P16_POLY_STATS[horizon])
_widths = {len(P16_MOMENT[h]) for h in range(12)}
assert len(_widths) == 1, f"horizon 별 설계행렬 폭이 다르다: {_widths}"
print(f"정규방정식 구성 {time.perf_counter() - _started:.1f}초 (12 horizon, 폭 {_widths.pop()})")


def p16_predict(designs, weights, last_wind):
    prediction = np.zeros((len(last_wind), 12), np.float64)
    for horizon in range(12):
        prediction[:, horizon] = designs[horizon] @ weights[horizon] + P16_INTERCEPT[horizon]
    return np.clip(prediction + last_wind[:, None], CLIP_LOW, CLIP_HIGH)


P16_SWEEP = []
for alpha in P16_ALPHAS:
    weights = {h: p16_ridge(P16_GRAM[h], P16_MOMENT[h], alpha) for h in range(12)}
    score, _ = official_rmse(val_targets, p16_predict(P16_VAL_DESIGN, weights, val_wind[:, -1]))
    P16_SWEEP.append({"alpha": alpha, "validation_rmse": score})
    print(f"  alpha={alpha:>9.1f}  validation 공식 RMSE = {score:8.3f} km/s")

P16_SWEEP = pd.DataFrame(P16_SWEEP)
P16_BEST_ALPHA = float(P16_SWEEP.loc[P16_SWEEP.validation_rmse.idxmin(), "alpha"])
P16_WEIGHTS = {h: p16_ridge(P16_GRAM[h], P16_MOMENT[h], P16_BEST_ALPHA) for h in range(12)}
print(f"\n선택된 alpha = {P16_BEST_ALPHA:g}")
if P16_BEST_ALPHA in (min(P16_ALPHAS), max(P16_ALPHAS)):
    print("  주의: alpha 가 격자 끝에서 골라졌다. P16_ALPHAS 범위를 넓힐 것.")
print("  alpha 를 validation 으로 골랐으므로 아래 RMSE 는 낙관적으로 편향된다.")
print("  조합 간 비교에는 문제없지만, 절대 성능은 test 로만 판단할 것.")

## 10. Validation 평가

In [ ]:
# ===================== P16: Validation 평가 ================================
validation_prediction = p16_predict(P16_VAL_DESIGN, P16_WEIGHTS, val_wind[:, -1])
model_score, _ = official_rmse(val_targets, validation_prediction)
validation_metrics = metrics_by_horizon(val_targets, validation_prediction, val_persistence)
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)

print(f"공식 RMSE (mean of horizon RMSE) : {model_score:8.3f} km/s")
print(f"pooled RMSE (전체 원소)          : {pooled_rmse(val_targets, validation_prediction):8.3f} km/s")
print(f"persistence 공식 RMSE            : {persistence_score:8.3f} km/s")
print(f"persistence 대비 개선             : {persistence_score - model_score:8.3f} km/s"
      f"  ({(persistence_score - model_score) / persistence_score:.1%})")
if model_score >= persistence_score:
    print("\n>>> persistence 미달. 이 피처 조합에는 선형·다항 범위의 신호가 없다.")

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(validation_metrics.horizon_h, validation_metrics.rmse, marker="o", label="P16 ridge")
axes[0].plot(validation_metrics.horizon_h, validation_metrics.persistence_rmse,
             marker="s", linestyle="--", label="persistence")
axes[0].set_xlabel("forecast horizon (h)"); axes[0].set_ylabel("RMSE (km/s)")
axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].semilogx(P16_SWEEP.alpha, P16_SWEEP.validation_rmse, marker="o")
axes[1].axvline(P16_BEST_ALPHA, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("ridge alpha"); axes[1].set_ylabel("validation 공식 RMSE")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ---- 어떤 항이 실제로 일하고 있는지 ------------------------------------------
# 설계행렬이 열별로 표준화되어 있으므로 계수 크기를 그대로 비교할 수 있다.
_core_names_kept = [name for name, keep in zip(P16_CORE_NAMES, P16_CORE_STATS[0][0]) if keep]
P16_TERM_NAMES = (
    [name for name, keep in zip(p16_polynomial_names(_core_names_kept, P16_DEGREE),
                                P16_POLY_STATS[0][0]) if keep]
    + P16_LINEAR_NAMES
)
assert len(P16_TERM_NAMES) == len(P16_WEIGHTS[0]), (len(P16_TERM_NAMES), len(P16_WEIGHTS[0]))
_importance = np.mean([np.abs(P16_WEIGHTS[h]) for h in range(12)], axis=0)
P16_TOP_TERMS = (pd.DataFrame({"term": P16_TERM_NAMES, "mean_abs_coef": _importance})
                 .sort_values("mean_abs_coef", ascending=False).head(20).reset_index(drop=True))
P16_TOP_TERMS.to_csv(OUTPUT_DIR / "top_terms.csv", index=False)
print("\n계수 크기 상위 20항 (12 horizon 평균, 표준화 기준):")
print(P16_TOP_TERMS.to_string(index=False, float_format=lambda value: f"{value:8.3f}"))

print("\nP16 switches:", P16_ACTIVE_SWITCHES)
print(f"P16 degree={P16_DEGREE} alpha={P16_BEST_ALPHA:g} width={P16_WIDTH}")
print(f"P16 OFFICIAL_VALIDATION_RMSE = {model_score:.4f} km/s")
validation_metrics

## 11. Test 추론 · 제출 파일 생성

In [ ]:
# ===================== P16: Test 추론 · 제출 파일 ==========================
_test_design = {}
for horizon in range(12):
    _test_design[horizon], _ = p16_design("test", horizon, P16_POLY_STATS[horizon])
test_prediction = p16_predict(_test_design, P16_WEIGHTS, test_wind[:, -1])

assert test_prediction.shape == (len(test_inputs), 12)
assert np.isfinite(test_prediction).all()

submission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", test_inputs.sample_id.tolist())
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

# 규정: "code.ipynb 에서 model.pth 를 불러와 추론이 가능해야 함".
# 릿지는 가중치·표준화 통계·절편이 전부이므로 그대로 직렬화한다.
checkpoint = {
    "kind": "p16_polynomial_ridge",
    "degree": P16_DEGREE,
    "alpha": P16_BEST_ALPHA,
    "switches": P16_ACTIVE_SWITCHES,
    "summaries": list(P16_SUMMARIES),
    "term_names": P16_TERM_NAMES,
    "intercept": {h: P16_INTERCEPT[h] for h in range(12)},
    "weights": {h: P16_WEIGHTS[h] for h in range(12)},
    "core_stats": {h: P16_CORE_STATS[h] for h in range(12)},
    "poly_stats": {h: P16_POLY_STATS[h] for h in range(12)},
    "linear_stats": P16_LINEAR_STATS,
    "clip": (CLIP_LOW, CLIP_HIGH),
    "validation_rmse": model_score,
}
checkpoint_path = OUTPUT_DIR / "p16_ridge.pth"
torch.save(checkpoint, checkpoint_path)
shutil.copyfile(checkpoint_path, SUBMISSION_DIR / "model.pth")

# 저장한 파일에서 되읽어 같은 예측이 나오는지 실제로 확인한다.
reloaded = torch.load(SUBMISSION_DIR / "model.pth", map_location="cpu", weights_only=False)
_reloaded_prediction = p16_predict(_test_design, reloaded["weights"], test_wind[:, -1])
assert np.allclose(_reloaded_prediction, test_prediction), "model.pth 재현 실패"
print("reloaded from submission/model.pth — test 예측 재현 확인")

print("saved:", (SUBMISSION_DIR / "submission.csv").resolve(), submission.shape)
print(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))
del _test_design
gc.collect()
submission.head()

## 12. 제출 전 체크리스트

In [ ]:
EXPECTED_TEST_ROWS = 3868

print("=== 제출 점검 ===")
ok = True
for name in ["code.ipynb", "model.pth", "submission.csv"]:
    path = SUBMISSION_DIR / name
    if path.exists():
        print(f"  [O] {name:16s} {path.stat().st_size / 1024 ** 2:8.2f} MiB")
    else:
        print(f"  [X] {name:16s} 없음")
        ok = False

check = pd.read_csv(SUBMISSION_DIR / "submission.csv")
print(f"\n  행 수      : {len(check):,} (기대 {EXPECTED_TEST_ROWS:,})",
      "OK" if len(check) == EXPECTED_TEST_ROWS else "<-- 불일치")
print(f"  컬럼       : {check.columns.tolist() == ['sample_id'] + TARGET_COLUMNS}")
print(f"  결측       : {int(check[TARGET_COLUMNS].isna().sum().sum())}")
print(f"  sample_id  : 유일={check.sample_id.is_unique}, "
      f"test_ids 일치={sorted(check.sample_id) == sorted(test_ids.sample_id)}")
print(f"  값 범위    : [{check[TARGET_COLUMNS].to_numpy().min():.1f}, "
      f"{check[TARGET_COLUMNS].to_numpy().max():.1f}] km/s")

extra = [p.name for p in SUBMISSION_DIR.iterdir()
         if p.name not in {"code.ipynb", "model.pth", "submission.csv"}]
if extra:
    print(f"\n  주의: 불필요한 파일 -> {extra}")